# LeetCode 556 — Next Greater Element III bằng Python

Link đề: https://leetcode.com/problems/next-greater-element-iii/ · Độ khó: **Medium** · Chủ đề: Math, Two Pointers, String

Skeleton LeetCode:

```python
class Solution:
    def nextGreaterElement(self, n: int) -> int:
        ...
```

Ghi chú: tên bài dễ gây nhầm với "Next Greater Element I/II" (dùng ngăn xếp đơn điệu trên mảng). Bài này **không** phải vậy: nó thực chất là bài **"hoán vị kế tiếp" (next permutation)** áp dụng lên các chữ số của một số nguyên, kèm một cái bẫy tràn số 32-bit. Notebook đi từ cách "thử từng số" ngây thơ, qua sinh hoán vị, qua đếm chữ số tham lam, rồi tới thuật toán next permutation chuẩn `O(d)` (với `d` là số chữ số).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho số nguyên dương `n`, tìm số **nhỏ nhất** lớn hơn `n` mà được tạo từ **đúng các chữ số của `n`** (xếp lại thứ tự), và phải nằm trong khoảng số nguyên 32-bit có dấu; không có thì trả `-1`.
2. Nắm các công cụ Python cần dùng: đổi qua lại `int` / `str` / `list` các chữ số, `sorted`, `Counter`, `itertools.permutations`, cắt lát (slicing) và đảo `[::-1]`, hoán đổi `a[i], a[j] = a[j], a[i]`.
3. Đi từ **thử từng số `n+1, n+2, ...`** → **sinh mọi hoán vị** → **đếm chữ số tham lam** → **next permutation hai lần quét**, và hiểu vì sao cách cuối đúng và nhanh.
4. Hiểu mấu chốt: **hậu tố không tăng (từ trái sang phải) thì đã là lớn nhất có thể**, nên ta phải tăng chữ số ngay trước hậu tố đó (pivot) lên thành chữ số lớn hơn **nhỏ nhất** có trong hậu tố, rồi xếp phần còn lại **tăng dần**.
5. Tránh các lỗi kinh điển: dùng `>` thay vì `>=` khi có chữ số trùng, chọn nhầm chữ số để hoán đổi, quên đảo hậu tố, và quên kiểm tra tràn `2**31 - 1`.

## 2. Đề bài & ví dụ

Cho một số nguyên dương `n`. Hãy tìm **số nguyên nhỏ nhất** thỏa mãn đồng thời:

- Có **đúng các chữ số** như `n` (cùng tập chữ số, cùng số lần xuất hiện, chỉ khác thứ tự), và
- **Lớn hơn `n`**.

Nếu không tồn tại số như vậy, trả về `-1`. Lưu ý: kết quả phải **vừa trong số nguyên 32-bit có dấu** (tối đa `2**31 - 1 = 2147483647`). Nếu có đáp án hợp lệ về mặt toán học nhưng **vượt** giới hạn này thì cũng trả về `-1`.

**Ví dụ 1**

- Input: `n = 12`
- Output: `21`

**Ví dụ 2**

- Input: `n = 21`
- Output: `-1` (chỉ có hai số `12` và `21`; không có số nào lớn hơn `21`)

**Ràng buộc:**

- `1 <= n <= 2**31 - 1`

Ví dụ tự thêm (không có trong đề gốc, để làm quen): `n = 230241` thì đáp án là `230412`.

In [1]:
from typing import List
from collections import Counter
from itertools import permutations
import random
import time

INT_MAX = 2 ** 31 - 1          # 2147483647, số nguyên 32-bit có dấu lớn nhất

examples = [(12, 21), (21, -1), (230241, 230412)]
for n, expected in examples:
    print("n =", n, "| số chữ số:", len(str(n)), "| kỳ vọng:", expected)
print("INT_MAX =", INT_MAX, "| có", len(str(INT_MAX)), "chữ số")

n = 12 | số chữ số: 2 | kỳ vọng: 21
n = 21 | số chữ số: 2 | kỳ vọng: -1
n = 230241 | số chữ số: 6 | kỳ vọng: 230412
INT_MAX = 2147483647 | có 10 chữ số


## 3. Kiến thức nền

### 3.1. Số, chuỗi và danh sách chữ số

Muốn thao tác từng chữ số của số nguyên, cách dễ nhất trong Python là đổi sang chuỗi `str(n)`, rồi sang danh sách `list(...)` các ký tự. Chiều ngược lại dùng `"".join(...)` rồi `int(...)`.

Một điểm hay: **hai chuỗi chữ số có cùng độ dài thì so sánh theo thứ tự từ điển cho kết quả giống hệt so sánh số**. Ví dụ `"21" > "12"` đúng và `21 > 12` cũng đúng. Nhưng nếu khác độ dài thì không: `"9" > "10"` là `True` (vì `"9" > "1"`) trong khi `9 > 10` là `False`. Vì mọi hoán vị của `n` đều có cùng số chữ số với `n`, ta yên tâm so sánh chuỗi cùng độ dài.

In [2]:
n = 230241
cs = list(str(n))
print(cs)                       # ['2', '3', '0', '2', '4', '1']
print(int("".join(cs)))         # 230241

print("21" > "12", 21 > 12)     # cùng độ dài: giống nhau
print("9" > "10", 9 > 10)       # khác độ dài: KHÁC nhau -> chỉ so sánh chuỗi cùng độ dài

['2', '3', '0', '2', '4', '1']
230241
True True
True False


### 3.2. "Cùng các chữ số" nghĩa là gì: `sorted` và `Counter`

Hai số có **cùng tập chữ số (có tính số lần lặp)** khi và chỉ khi sắp xếp các chữ số của chúng ra thì được cùng một dãy. Cũng có thể dùng `Counter` (bộ đếm) để so sánh số lần xuất hiện của từng chữ số.

In [3]:
a, b, c = 1243, 3124, 1244
print(sorted(str(a)), sorted(str(b)), sorted(str(c)))
print(sorted(str(a)) == sorted(str(b)))      # True: cùng chữ số 1,2,3,4
print(sorted(str(a)) == sorted(str(c)))      # False: c có hai chữ số 4
print(Counter("112"), Counter("121") == Counter("112"))

['1', '2', '3', '4'] ['1', '2', '3', '4'] ['1', '2', '4', '4']
True
False
Counter({'1': 2, '2': 1}) True


### 3.3. Hoán vị bằng `itertools.permutations`

`permutations("123")` sinh mọi cách xếp thứ tự (hoán vị). Nếu có chữ số **lặp** thì nó vẫn coi các vị trí là khác nhau nên sinh ra các hoán vị **trùng giá trị**: `"112"` có `3! = 6` hoán vị nhưng chỉ có `3` chuỗi khác nhau. Dùng `set(...)` để loại trùng.

In [4]:
perms = list(permutations("123"))
print(len(perms), perms[:3])
p112 = list(permutations("112"))
print(len(p112), "hoán vị, nhưng chỉ", len(set(p112)), "khác nhau:", sorted("".join(p) for p in set(p112)))

6 [('1', '2', '3'), ('1', '3', '2'), ('2', '1', '3')]
6 hoán vị, nhưng chỉ 3 khác nhau: ['112', '121', '211']


### 3.4. Cắt lát, đảo, hoán đổi

Ba thao tác trên danh sách sẽ dùng liên tục:

- `cs[i + 1:]` lấy **hậu tố** (từ vị trí `i + 1` đến hết); `cs[::-1]` đảo ngược.
- Gán lát cắt `cs[i + 1:] = ...` thay cả hậu tố bằng nội dung mới.
- Hoán đổi hai phần tử không cần biến tạm: `cs[i], cs[j] = cs[j], cs[i]`.

In [5]:
cs = list("158476531")
i = 3
print("hậu tố sau i:", cs[i + 1:])
print("đảo hậu tố :", cs[i + 1:][::-1])
cs[i + 1:] = cs[i + 1:][::-1]         # gán lại hậu tố đã đảo
print(cs)
x = [10, 20, 30]
x[0], x[2] = x[2], x[0]
print(x)                               # [30, 20, 10]

hậu tố sau i: ['7', '6', '5', '3', '1']
đảo hậu tố : ['1', '3', '5', '6', '7']
['1', '5', '8', '4', '1', '3', '5', '6', '7']
[30, 20, 10]


### 3.5. Giới hạn 32-bit

Số nguyên 32-bit có dấu có giá trị lớn nhất `2**31 - 1 = 2147483647` (10 chữ số). Python không tràn số (số nguyên to tùy ý) nên **ta phải tự kiểm tra** `ans > 2**31 - 1`. Hoán vị tăng lên có thể làm vượt giới hạn dù `n` hợp lệ, ví dụ `n = 1999999999` (hợp lệ) nhưng hoán vị kế tiếp là `9199999999` (lớn hơn rất nhiều so với `2147483647`).

In [6]:
print(2 ** 31 - 1)
n = 1999999999
print(n <= INT_MAX)                    # True: n hợp lệ
print(9199999999 <= INT_MAX)           # False: hoán vị kế tiếp tràn 32-bit

2147483647
True
False


### 3.6. Thứ tự từ điển (lexicographic) và "hoán vị kế tiếp"

Nếu liệt kê **mọi hoán vị khác nhau** của các chữ số theo thứ tự tăng dần (thứ tự từ điển) thì "số lớn hơn `n` nhỏ nhất" chính là **phần tử đứng ngay sau `n`** trong danh sách đó. Đó là lý do bài này gọi là *next permutation*. Nếu `n` đã đứng cuối danh sách thì không có đáp án (`-1`).

In [7]:
def all_perms_sorted(n):
    return sorted(set(int("".join(p)) for p in permutations(str(n))))

lst = all_perms_sorted(1243)
print(lst)
k = lst.index(1243)
print("đứng ngay sau 1243 là:", lst[k + 1])
print("21 ->", all_perms_sorted(21), "| 21 đứng cuối nên đáp án -1")

[1234, 1243, 1324, 1342, 1423, 1432, 2134, 2143, 2314, 2341, 2413, 2431, 3124, 3142, 3214, 3241, 3412, 3421, 4123, 4132, 4213, 4231, 4312, 4321]
đứng ngay sau 1243 là: 1324
21 -> [12, 21] | 21 đứng cuối nên đáp án -1


## 4. Ý tưởng nền tảng / trực giác (tính tay)

Hãy làm tay với `n = 1243`. Mọi hoán vị tăng dần là (như cell trên): `1234, 1243, 1324, 1342, 1423, 1432, 2134, ...`. Số kế tiếp của `1243` là `1324`. Nhận xét:

- Hai chữ số đầu `1 2` đổi thành `1 3`: ta **không** động tới chữ số đầu tiên `1`, chỉ thay đổi từ chữ số thứ hai trở đi.
- Phần còn lại sau chữ số được tăng `(3)` thì xếp **nhỏ nhất có thể**: `2, 4` thành `2 4`, tức `1 3 2 4`.

Quy tắc tổng quát suy ra từ tính toán tay:

1. Để số mới **lớn hơn `n` nhưng tăng ít nhất**, ta muốn giữ **tiền tố giống nhau càng dài càng tốt** (vì chữ số bên trái có trọng số lớn hơn). Vậy vị trí bị thay đổi phải **càng bên phải càng tốt**.
2. Tại vị trí đó, chữ số phải **tăng** (nếu không số mới sẽ nhỏ hơn).
3. Sau đó, các chữ số bên phải vị trí này sắp xếp sao cho số **nhỏ nhất**: **tăng dần**.

Câu hỏi còn lại: vị trí "càng bên phải càng tốt mà vẫn tăng được" là vị trí nào? Xem ví dụ `n = 4321`: các chữ số giảm dần từ trái sang phải. Không thể xếp lại phần nào để được số lớn hơn vì `4321` đã là số lớn nhất của các chữ số `{1,2,3,4}`. Câu trả lời: vị trí cần tìm là **vị trí đầu tiên (tính từ phải) mà chữ số nhỏ hơn chữ số ngay bên phải nó**.

In [8]:
# Với mỗi n, tìm tay: vị trí i lớn nhất sao cho cs[i] < cs[i+1]
for n in [1243, 4321, 230241, 115, 11]:
    cs = list(str(n))
    pos = -1
    for i in range(len(cs) - 2, -1, -1):
        if cs[i] < cs[i + 1]:
            pos = i
            break
    print(n, "-> vị trí tăng được:", pos, "(-1 nghĩa là không có: n đã là hoán vị lớn nhất)")

1243 -> vị trí tăng được: 1 (-1 nghĩa là không có: n đã là hoán vị lớn nhất)
4321 -> vị trí tăng được: -1 (-1 nghĩa là không có: n đã là hoán vị lớn nhất)
230241 -> vị trí tăng được: 3 (-1 nghĩa là không có: n đã là hoán vị lớn nhất)
115 -> vị trí tăng được: 1 (-1 nghĩa là không có: n đã là hoán vị lớn nhất)
11 -> vị trí tăng được: -1 (-1 nghĩa là không có: n đã là hoán vị lớn nhất)


## 5. Cách 1 — Brute Force: thử từng số `n + 1, n + 2, ...`

### Ý tưởng

Đọc đề thật thà: "số nhỏ nhất lớn hơn `n` có cùng chữ số". Vậy ta thử lần lượt `m = n + 1, n + 2, ...`, kiểm tra `m` có cùng tập chữ số với `n` không (so `sorted(str(m)) == sorted(str(n))`). Số đầu tiên thỏa là đáp án (vì ta duyệt tăng dần). Khi `m` có nhiều chữ số hơn `n` (đạt `10 ** d`) thì dừng, trả `-1` (mọi hoán vị phải có đúng `d` chữ số). Đồng thời dừng nếu `m > 2**31 - 1`.

### Code

In [9]:
def next_greater_scan(n):
    target = sorted(str(n))
    d = len(target)
    m = n + 1
    while m < 10 ** d and m <= INT_MAX:
        if sorted(str(m)) == target:
            return m
        m += 1
    return -1

for n, expected in examples:
    print(n, "->", next_greater_scan(n), "(kỳ vọng", expected, ")")

12 -> 21 (kỳ vọng 21 )
21 -> -1 (kỳ vọng -1 )
230241 -> 230412 (kỳ vọng 230412 )


### Độ phức tạp và nút thắt

- Time: trong trường hợp xấu phải thử gần `10^d` số, mỗi số tốn `O(d log d)` để sắp xếp: khoảng `O(10^d · d log d)`. Với `d = 10` là hàng chục tỉ phép thử — không thể chạy.
- Space: `O(d)`.

**Nút thắt:** ta thử cả những số rất xa nhau mà **không biết** chúng gần hay xa đáp án. Hãy xem số bước thử tăng nhanh thế nào khi số chữ số tăng.

In [10]:
def count_scan_steps(n):
    target = sorted(str(n)); d = len(target); m = n + 1; steps = 0
    while m < 10 ** d and m <= INT_MAX:
        steps += 1
        if sorted(str(m)) == target:
            return steps
        m += 1
    return steps

for n in [12, 1243, 12443, 124432, 1244332]:
    t0 = time.perf_counter()
    s = count_scan_steps(n)
    print(f"n={n}: {s} bước thử, {time.perf_counter() - t0:.4f}s")

n=12: 9 bước thử, 0.0000s
n=1243: 81 bước thử, 0.0001s
n=12443: 801 bước thử, 0.0004s
n=124432: 7812 bước thử, 0.0032s
n=1244332: 78012 bước thử, 0.0756s


## 6. Ý tưởng thuật toán / bước đột phá

Từ trực giác ở mục 4, ta không cần thử số nào cả. Ta có thể **dựng thẳng** đáp án từ các chữ số. Bước đột phá là nhận ra cấu trúc **hậu tố không tăng**:

> Gọi `cs` là dãy chữ số của `n`. Tìm **pivot** `i` lớn nhất sao cho `cs[i] < cs[i + 1]`. Khi đó đoạn `cs[i + 1:]` **không tăng** (mỗi chữ số `>=` chữ số bên phải nó), tức hậu tố đã là cách xếp **lớn nhất** của chính nó. Do đó không cách xếp lại hậu tố nào cho số lớn hơn: ta bắt buộc phải tăng chữ số tại `i`.

Thuật toán `next permutation` gồm 4 bước:

1. **Tìm pivot `i`:** từ phải sang trái, tìm `i` đầu tiên có `cs[i] < cs[i + 1]`. Không có thì `-1`.
2. **Tìm `j`:** từ phải sang trái, tìm `j` đầu tiên có `cs[j] > cs[i]`. Vì hậu tố không tăng, đây chính là chữ số **nhỏ nhất lớn hơn** `cs[i]` trong hậu tố.
3. **Hoán đổi** `cs[i]` và `cs[j]`.
4. **Đảo ngược** hậu tố `cs[i + 1:]`. Sau bước 3 hậu tố vẫn không tăng, nên đảo ngược nó thành **tăng dần** = nhỏ nhất.

Thử với `n = 158476531`: pivot là chữ số `4` (vì `4 < 7`, còn `7 >= 6 >= 5 >= 3 >= 1`); chữ số nhỏ nhất lớn hơn `4` trong hậu tố `7 6 5 3 1` là `5`; hoán đổi được `1 5 8 5 7 6 4 3 1`; đảo hậu tố `7 6 4 3 1` thành `1 3 4 6 7`; kết quả `158513467`.

In [11]:
cs = list("158476531")
i = len(cs) - 2
while i >= 0 and cs[i] >= cs[i + 1]:
    i -= 1
print("pivot i =", i, "chữ số", cs[i], "| hậu tố:", cs[i + 1:])
j = len(cs) - 1
while cs[j] <= cs[i]:
    j -= 1
print("j =", j, "chữ số", cs[j])
cs[i], cs[j] = cs[j], cs[i]
print("sau hoán đổi:", "".join(cs))
cs[i + 1:] = cs[i + 1:][::-1]
print("sau đảo hậu tố:", "".join(cs))
print("kết quả:", "".join(cs), "| Cách 1 phải thử", int("".join(cs)) - 158476531, "số mới tới được đáp án này")

pivot i = 3 chữ số 4 | hậu tố: ['7', '6', '5', '3', '1']
j = 6 chữ số 5
sau hoán đổi: 158576431
sau đảo hậu tố: 158513467
kết quả: 158513467 | Cách 1 phải thử 36936 số mới tới được đáp án này


## 7. Các cách tiếp theo

Tổng cộng ta có 4 cách khác nhau thực sự về ý tưởng:

| Cách | Ý tưởng |
|---|---|
| 1 | Thử `n+1, n+2, ...` (mục 5) |
| 2 | Sinh toàn bộ hoán vị, lọc lấy số nhỏ nhất lớn hơn `n` |
| 3 | Duyệt từ phải sang, dùng **mảng đếm 10 chữ số** của hậu tố, dựng đáp án tham lam |
| 4 | **Next permutation**: pivot, hoán đổi, đảo hậu tố (cách nộp) |

### Cách 2 — Sinh mọi hoán vị

**Giải thích từng đoạn:**

- `set(permutations(str(n)))`: tạo mọi chuỗi chữ số có thể, loại trùng.
- Với mỗi hoán vị, đổi sang số `m`; giữ lại những `m > n` và `m <= INT_MAX`.
- Chọn `m` nhỏ nhất trong số giữ lại; không có thì `-1`.

### Code demo

In [12]:
def next_greater_perms(n):
    best = -1
    for p in set(permutations(str(n))):
        m = int("".join(p))
        if n < m <= INT_MAX and (best == -1 or m < best):
            best = m
    return best

for n, expected in examples:
    print(n, "->", next_greater_perms(n), "(kỳ vọng", expected, ")")

12 -> 21 (kỳ vọng 21 )
21 -> -1 (kỳ vọng -1 )
230241 -> 230412 (kỳ vọng 230412 )


### Code nộp LeetCode (Cách 2)

```python
from itertools import permutations

class Solution:
    def nextGreaterElement(self, n: int) -> int:
        best = -1
        for p in set(permutations(str(n))):
            m = int("".join(p))
            if n < m <= 2 ** 31 - 1 and (best == -1 or m < best):
                best = m
        return best
```

### Độ phức tạp

- Time: `O(d! · d)` trong trường hợp xấu (các chữ số đôi một khác nhau). `d = 10` cho `3 628 800` hoán vị, mỗi cái ghép chuỗi `O(d)` — chậm, tốn bộ nhớ vì `set`.
- Space: `O(d! · d)` cho tập hoán vị.

Cách này tốt hơn Cách 1 (`d!` so với `10^d`) và rất tiện làm **oracle** (đáp án đối chiếu) để kiểm thử, nhưng vẫn quá chậm cho `d = 10` ở các test khó.

In [13]:
for n in [1243, 12443, 124433, 12443322]:
    t0 = time.perf_counter()
    r = next_greater_perms(n)
    print(f"n={n}: {r}  ({len(set(permutations(str(n))))} hoán vị khác nhau, {time.perf_counter() - t0:.3f}s)")

n=1243: 1324  (24 hoán vị khác nhau, 0.000s)
n=12443: 13244  (60 hoán vị khác nhau, 0.000s)
n=124433: 132344  (180 hoán vị khác nhau, 0.000s)
n=12443322: 13222344  (1680 hoán vị khác nhau, 0.006s)


### Cách 3 — Đếm chữ số của hậu tố, dựng đáp án tham lam

**Giải thích từng đoạn:**

- Duyệt `i` từ **phải sang trái**, giữ mảng `cnt[0..9]` đếm các chữ số **bên phải** `i`.
- Tại `i` với chữ số `d = cs[i]`: tìm chữ số `e` **nhỏ nhất** thỏa `e > d` và `cnt[e] > 0`. Duyệt `e` từ `d + 1` đến `9`, gặp `cnt[e] > 0` là dừng.
- Nếu có `e`: đáp án = `tiền_tố(cs[:i]) + e + (các chữ số còn lại xếp tăng dần)`. "Các chữ số còn lại" là hậu tố **bỏ một chữ `e`, thêm chữ `d`** (vì `d` bị đẩy sang phải). Xếp tăng dần bằng cách in `cnt[0]` số `0`, `cnt[1]` số `1`, ...
- Nếu không có `e` thì chữ số `d` gia nhập hậu tố: `cnt[d] += 1`, rồi tiếp tục sang trái.
- Hết vòng lặp mà chưa dựng được: `-1`.

Khác với Cách 4: Cách 3 không cần biết hậu tố "không tăng"; nó dùng thẳng mảng đếm nên tự đúng và dễ hiểu. Hai cách cho cùng kết quả vì chúng cùng chọn pivot (vị trí phải nhất có chữ số lớn hơn trong hậu tố).

### Code demo

In [14]:
def next_greater_count(n):
    s = str(n)
    cnt = [0] * 10                       # cnt[c] = số lần chữ số c xuất hiện bên phải vị trí i
    for i in range(len(s) - 1, -1, -1):
        d = int(s[i])
        for e in range(d + 1, 10):       # chữ số nhỏ nhất > d còn có trong hậu tố
            if cnt[e]:
                cnt[e] -= 1              # lấy e ra đặt tại vị trí i
                cnt[d] += 1              # d bị đẩy vào phần đuôi
                tail = "".join(str(k) * cnt[k] for k in range(10))   # đuôi tăng dần
                ans = int(s[:i] + str(e) + tail)
                return ans if ans <= INT_MAX else -1
        cnt[d] += 1
    return -1

for n, expected in examples:
    print(n, "->", next_greater_count(n), "(kỳ vọng", expected, ")")
print(next_greater_count(158476531), next_greater_count(1999999999))

12 -> 21 (kỳ vọng 21 )
21 -> -1 (kỳ vọng -1 )
230241 -> 230412 (kỳ vọng 230412 )
158513467 -1


### Code nộp LeetCode (Cách 3)

```python
class Solution:
    def nextGreaterElement(self, n: int) -> int:
        s = str(n)
        cnt = [0] * 10
        for i in range(len(s) - 1, -1, -1):
            d = int(s[i])
            for e in range(d + 1, 10):
                if cnt[e]:
                    cnt[e] -= 1
                    cnt[d] += 1
                    tail = "".join(str(k) * cnt[k] for k in range(10))
                    ans = int(s[:i] + str(e) + tail)
                    return ans if ans <= 2 ** 31 - 1 else -1
            cnt[d] += 1
        return -1
```

### Độ phức tạp

- Time: `O(10 · d)` = `O(d)`: mỗi vị trí duyệt tối đa 9 chữ số `e`, rồi dựng đuôi `O(d)` đúng một lần.
- Space: `O(d)` (chuỗi) cộng mảng đếm `O(10)`.

### Cách 4 — Next permutation (pivot, hoán đổi, đảo hậu tố)

**Giải thích từng đoạn** (đã nói ở mục 6):

- `i`: lùi từ `d - 2` trong khi `cs[i] >= cs[i + 1]`. Dùng `>=` (không phải `>`) để **bỏ qua cả chữ số bằng nhau**: `1 1` không thể tăng.
- Nếu `i < 0` thì toàn bộ dãy không tăng: `-1`.
- `j`: lùi từ cuối trong khi `cs[j] <= cs[i]`: dừng ở chữ số đầu tiên **lớn hơn hẳn** `cs[i]`.
- Hoán đổi, đảo hậu tố, ghép lại, kiểm tra `> 2**31 - 1`.

### Code demo

In [15]:
def next_greater_perm(n):
    cs = list(str(n))
    i = len(cs) - 2
    while i >= 0 and cs[i] >= cs[i + 1]:      # tìm pivot (>= để xử lý chữ số trùng)
        i -= 1
    if i < 0:
        return -1                              # toàn dãy không tăng: đã là lớn nhất
    j = len(cs) - 1
    while cs[j] <= cs[i]:                      # chữ số nhỏ nhất lớn hơn cs[i] trong hậu tố
        j -= 1
    cs[i], cs[j] = cs[j], cs[i]
    cs[i + 1:] = cs[i + 1:][::-1]              # đảo hậu tố -> tăng dần
    ans = int("".join(cs))
    return ans if ans <= INT_MAX else -1

for n, expected in examples:
    print(n, "->", next_greater_perm(n), "(kỳ vọng", expected, ")")

12 -> 21 (kỳ vọng 21 )
21 -> -1 (kỳ vọng -1 )
230241 -> 230412 (kỳ vọng 230412 )


### Code nộp LeetCode (Cách 4)

```python
class Solution:
    def nextGreaterElement(self, n: int) -> int:
        cs = list(str(n))
        i = len(cs) - 2
        while i >= 0 and cs[i] >= cs[i + 1]:
            i -= 1
        if i < 0:
            return -1
        j = len(cs) - 1
        while cs[j] <= cs[i]:
            j -= 1
        cs[i], cs[j] = cs[j], cs[i]
        cs[i + 1:] = cs[i + 1:][::-1]
        ans = int("".join(cs))
        return ans if ans <= 2 ** 31 - 1 else -1
```

### Độ phức tạp

- Time: `O(d)` — hai lần quét từ phải, một lần đảo hậu tố, một lần ghép chuỗi. Với `n <= 2**31 - 1` thì `d <= 10` nên thực tế là hằng số.
- Space: `O(d)` cho danh sách chữ số.

## 8. TRACE từng bước

Hàm `trace_next_greater(n)` dưới đây in từng bước của Cách 4: mỗi lần lùi `i`, mỗi lần lùi `j`, khi hoán đổi, khi đảo hậu tố và khi kiểm tra tràn.

Bảng trạng thái cho `n = 158476531` (chỉ số 0..8):

| Bước | Hành động | `cs` | Ghi chú |
|---|---|---|---|
| 0 | khởi tạo | `1 5 8 4 7 6 5 3 1` | `i = 7` |
| 1 | lùi `i` (7, 6, 5, 4) | giữ nguyên | `3>=1`, `5>=3`, `6>=5`, `7>=6` |
| 2 | dừng ở `i = 3` | giữ nguyên | `cs[3]=4 < cs[4]=7`: pivot |
| 3 | tìm `j` từ cuối | giữ nguyên | `1<=4`, `3<=4` lùi; `5>4` dừng: `j = 6` |
| 4 | hoán đổi `cs[3], cs[6]` | `1 5 8 5 7 6 4 3 1` | hậu tố `7 6 4 3 1` vẫn không tăng |
| 5 | đảo `cs[4:]` | `1 5 8 5 1 3 4 6 7` | kết quả `158513467` |

In [16]:
def trace_next_greater(n):
    cs = list(str(n))
    print(f"n = {n} | chữ số = {cs}")
    i = len(cs) - 2
    print("-- Bước 1: tìm pivot i (lùi khi cs[i] >= cs[i+1])")
    while i >= 0 and cs[i] >= cs[i + 1]:
        print(f"   i={i}: cs[{i}]={cs[i]} >= cs[{i + 1}]={cs[i + 1]} -> hậu tố chưa tăng được, lùi i")
        i -= 1
    if i < 0:
        print("   i < 0: toàn bộ dãy không tăng -> trả -1")
        return -1
    print(f"   i={i}: cs[{i}]={cs[i]} < cs[{i + 1}]={cs[i + 1]} -> pivot = {i}, hậu tố = {cs[i + 1:]}")
    j = len(cs) - 1
    print("-- Bước 2: tìm j (lùi khi cs[j] <= cs[i])")
    while cs[j] <= cs[i]:
        print(f"   j={j}: cs[{j}]={cs[j]} <= {cs[i]} -> lùi j")
        j -= 1
    print(f"   j={j}: cs[{j}]={cs[j]} > {cs[i]} -> chọn j = {j}")
    cs[i], cs[j] = cs[j], cs[i]
    print(f"-- Bước 3: hoán đổi cs[{i}] và cs[{j}] -> {''.join(cs)}")
    cs[i + 1:] = cs[i + 1:][::-1]
    print(f"-- Bước 4: đảo hậu tố cs[{i + 1}:] -> {''.join(cs)}")
    ans = int("".join(cs))
    if ans > INT_MAX:
        print(f"-- Bước 5: {ans} > {INT_MAX} -> tràn 32-bit, trả -1")
        return -1
    print(f"-- Bước 5: {ans} <= {INT_MAX} -> hợp lệ")
    return ans

print("=>", trace_next_greater(158476531))

n = 158476531 | chữ số = ['1', '5', '8', '4', '7', '6', '5', '3', '1']
-- Bước 1: tìm pivot i (lùi khi cs[i] >= cs[i+1])
   i=7: cs[7]=3 >= cs[8]=1 -> hậu tố chưa tăng được, lùi i
   i=6: cs[6]=5 >= cs[7]=3 -> hậu tố chưa tăng được, lùi i
   i=5: cs[5]=6 >= cs[6]=5 -> hậu tố chưa tăng được, lùi i
   i=4: cs[4]=7 >= cs[5]=6 -> hậu tố chưa tăng được, lùi i
   i=3: cs[3]=4 < cs[4]=7 -> pivot = 3, hậu tố = ['7', '6', '5', '3', '1']
-- Bước 2: tìm j (lùi khi cs[j] <= cs[i])
   j=8: cs[8]=1 <= 4 -> lùi j
   j=7: cs[7]=3 <= 4 -> lùi j
   j=6: cs[6]=5 > 4 -> chọn j = 6
-- Bước 3: hoán đổi cs[3] và cs[6] -> 158576431
-- Bước 4: đảo hậu tố cs[4:] -> 158513467
-- Bước 5: 158513467 <= 2147483647 -> hợp lệ
=> 158513467


In [17]:
print("=>", trace_next_greater(12443322))
print()
print("=>", trace_next_greater(4321))
print()
print("=>", trace_next_greater(1999999999))

n = 12443322 | chữ số = ['1', '2', '4', '4', '3', '3', '2', '2']
-- Bước 1: tìm pivot i (lùi khi cs[i] >= cs[i+1])
   i=6: cs[6]=2 >= cs[7]=2 -> hậu tố chưa tăng được, lùi i
   i=5: cs[5]=3 >= cs[6]=2 -> hậu tố chưa tăng được, lùi i
   i=4: cs[4]=3 >= cs[5]=3 -> hậu tố chưa tăng được, lùi i
   i=3: cs[3]=4 >= cs[4]=3 -> hậu tố chưa tăng được, lùi i
   i=2: cs[2]=4 >= cs[3]=4 -> hậu tố chưa tăng được, lùi i
   i=1: cs[1]=2 < cs[2]=4 -> pivot = 1, hậu tố = ['4', '4', '3', '3', '2', '2']
-- Bước 2: tìm j (lùi khi cs[j] <= cs[i])
   j=7: cs[7]=2 <= 2 -> lùi j
   j=6: cs[6]=2 <= 2 -> lùi j
   j=5: cs[5]=3 > 2 -> chọn j = 5
-- Bước 3: hoán đổi cs[1] và cs[5] -> 13443222
-- Bước 4: đảo hậu tố cs[2:] -> 13222344
-- Bước 5: 13222344 <= 2147483647 -> hợp lệ
=> 13222344

n = 4321 | chữ số = ['4', '3', '2', '1']
-- Bước 1: tìm pivot i (lùi khi cs[i] >= cs[i+1])
   i=2: cs[2]=2 >= cs[3]=1 -> hậu tố chưa tăng được, lùi i
   i=1: cs[1]=3 >= cs[2]=2 -> hậu tố chưa tăng được, lùi i
   i=0: cs[0]=4 >= c

Quan sát trace của `12443322`: pivot là chữ số `2` ở vị trí 1 (vì `2 < 4`); hậu tố `4 4 3 3 2 2` không tăng. Chữ số nhỏ nhất lớn hơn `2` trong hậu tố là `3` — và có **hai** chữ `3`; ta chọn cái **phải nhất**, nhờ đó sau khi hoán đổi hậu tố vẫn không tăng. Hoán đổi `2` (vị trí 1) với `3` (vị trí 5) được `13443222`, rồi đảo hậu tố `4 4 3 2 2 2` thành `2 2 2 3 4 4`, ghép lại `13222344`.

Trace của `1999999999`: pivot là `1` (vị trí 0), hoán đổi với `9` phải nhất, đảo hậu tố — ra `9199999999`, **lớn hơn** `2147483647` nên trả `-1`. Đây chính là cái bẫy tràn số của bài.

## 9. Code nộp LeetCode

Ta nộp Cách 4 (next permutation): ngắn, `O(d)`, không cần thư viện.

In [18]:
class Solution:
    def nextGreaterElement(self, n: int) -> int:
        cs = list(str(n))
        i = len(cs) - 2
        while i >= 0 and cs[i] >= cs[i + 1]:
            i -= 1
        if i < 0:
            return -1
        j = len(cs) - 1
        while cs[j] <= cs[i]:
            j -= 1
        cs[i], cs[j] = cs[j], cs[i]
        cs[i + 1:] = cs[i + 1:][::-1]
        ans = int("".join(cs))
        return ans if ans <= 2 ** 31 - 1 else -1


sol = Solution()
for n, expected in examples:
    got = sol.nextGreaterElement(n)
    print(n, "->", got, "| kỳ vọng:", expected, "| OK" if got == expected else "| SAI")

12 -> 21 | kỳ vọng: 21 | OK
21 -> -1 | kỳ vọng: -1 | OK
230241 -> 230412 | kỳ vọng: 230412 | OK


## 10. Vì sao đúng

Chứng minh cho Cách 4. Đặt `cs` là dãy chữ số, `i` là pivot (vị trí phải nhất có `cs[i] < cs[i + 1]`).

**(a) Phải thay đổi tại vị trí `<= i`.** Do `i` là vị trí phải nhất có `cs[i] < cs[i+1]`, nên `cs[i+1] >= cs[i+2] >= ... >= cs[d-1]`: hậu tố không tăng, tức đã là cách xếp lớn nhất của các chữ số hậu tố. Mọi cách giữ nguyên `cs[0..i]` và xếp lại hậu tố đều cho số `<= n`. Vậy đáp án phải khác `n` ở một vị trí `<= i`.

**(b) Thay đổi ở vị trí càng phải càng tốt.** Số lớn hơn `n` mà giữ tiền tố `cs[0..i-1]` (khác ở vị trí `i`) luôn **nhỏ hơn** số lớn hơn `n` mà đã khác ở vị trí `< i` (vì chữ số bên trái có trọng số lớn hơn). Nên ta chọn đổi tại đúng `i`.

**(c) Chữ số tại `i` phải tăng và tăng ít nhất.** Chữ số mới tại `i` phải lớn hơn `cs[i]` (để số lớn hơn `n`) và phải lấy từ các chữ số hậu tố (tập chữ số cố định). Muốn số nhỏ nhất, chọn chữ số **nhỏ nhất lớn hơn** `cs[i]`. Vì hậu tố không tăng, chữ số nhỏ nhất lớn hơn `cs[i]` nằm ở vị trí **phải nhất** thỏa `cs[j] > cs[i]` — đó là `j` mà ta quét tới.

**(d) Phần đuôi xếp nhỏ nhất.** Sau khi đặt chữ số mới tại `i`, các chữ số còn lại (hậu tố đã đổi `cs[j]` lấy `cs[i]`) phải xếp **tăng dần** mới nhỏ nhất. Hoán đổi giữ cho hậu tố vẫn không tăng: chữ số đứng trước `j` trong hậu tố đều `> cs[i]` hoặc bằng `cs[j]` hoặc lớn hơn (vì ta lấy `j` phải nhất thỏa `cs[j] > cs[i]` nên `cs[j-1] >= cs[j]`), còn chữ số đứng sau `j` đều `<= cs[i]` (nếu không `j` đã không phải phải nhất). Chữ số `cs[i]` vừa đặt vào vị trí `j` nằm giữa hai nhóm đó, nên dãy vẫn không tăng. Do vậy chỉ cần **đảo** là thành tăng dần, không cần sắp xếp `O(d log d)`.

**(e) Tràn 32-bit.** Đáp án toán học là duy nhất; ta chỉ việc so với `2**31 - 1`. Nếu đáp án nhỏ nhất đã vượt thì mọi số lớn hơn `n` còn lại (đều `>=` đáp án) cũng vượt, nên `-1` là đúng.

Dưới đây là kiểm chứng bằng thí nghiệm cho nhận định (d): sau hoán đổi, hậu tố vẫn không tăng.

In [19]:
rng = random.Random(556)
checked = 0
for _ in range(20000):
    d = rng.randint(2, 9)
    cs = [str(rng.randint(0, 4)) for _ in range(d)]       # ít loại chữ số để có nhiều chữ số trùng
    cs[0] = str(rng.randint(1, 4))
    i = d - 2
    while i >= 0 and cs[i] >= cs[i + 1]:
        i -= 1
    if i < 0:
        continue
    j = d - 1
    while cs[j] <= cs[i]:
        j -= 1
    cs[i], cs[j] = cs[j], cs[i]
    suffix = cs[i + 1:]
    assert all(suffix[k] >= suffix[k + 1] for k in range(len(suffix) - 1)), cs     # vẫn không tăng
    checked += 1
print("hậu tố sau hoán đổi luôn không tăng, đã kiểm", checked, "trường hợp")

hậu tố sau hoán đổi luôn không tăng, đã kiểm 16783 trường hợp


## 11. Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1 Thử `n+1, n+2...` | `O(10^d · d log d)` | `O(d)` |
| 2 Sinh hoán vị | `O(d! · d)` | `O(d! · d)` |
| 3 Đếm chữ số tham lam | `O(d)` (hằng số 10) | `O(d)` |
| 4 Next permutation | `O(d)` | `O(d)` |

Với `n <= 2**31 - 1` ta có `d <= 10`. Vì vậy Cách 4 gần như `O(1)`. Ta đo số thao tác của hai vòng quét để thấy chúng không quá `2d`.

In [20]:
def count_ops(n):
    cs = list(str(n)); d = len(cs); ops = 0
    i = d - 2
    while i >= 0 and cs[i] >= cs[i + 1]:
        i -= 1; ops += 1
    if i < 0:
        return ops
    j = d - 1
    while cs[j] <= cs[i]:
        j -= 1; ops += 1
    return ops + (d - 1 - i)          # cộng chi phí đảo hậu tố

rng = random.Random(1)
worst = 0
for _ in range(5000):
    n = rng.randint(1, INT_MAX)
    worst = max(worst, count_ops(n))
print("số thao tác lớn nhất trong 5000 số ngẫu nhiên:", worst, "(giới hạn lý thuyết khoảng 3d = 30)")

số thao tác lớn nhất trong 5000 số ngẫu nhiên: 18 (giới hạn lý thuyết khoảng 3d = 30)


## 12. Bộ test

Ta kiểm thử Cách 4 theo các nhóm: ví dụ của đề, edge case (một chữ số, toàn chữ số trùng, giảm dần, tràn 32-bit), phản ví dụ (các trường hợp mà code sai hay trượt), rồi đối chiếu **nhiều cách** trên input ngẫu nhiên có seed cố định.

In [21]:
# 1) Ví dụ của đề
assert Solution().nextGreaterElement(12) == 21
assert Solution().nextGreaterElement(21) == -1

# 2) Edge case
edge = {
    1: -1,                  # một chữ số
    9: -1,
    10: -1,                 # 10 -> chỉ có 01 (bắt đầu bằng 0, nhỏ hơn)
    11: -1,                 # toàn chữ số trùng
    111111: -1,
    101: 110,               # có chữ số 0
    1000000000: -1,         # giảm không tăng: 1 >= 0 >= 0 ...
    1000000001: 1000000010,
    4321: -1,               # giảm dần
    1234: 1243,
    # tràn 32-bit
    1999999999: -1,         # đáp án toán học 9199999999 > INT_MAX
    2147483647: -1,         # đáp án toán học 2147483674 > INT_MAX
    2147483467: 2147483476, # vẫn hợp lệ
    # chữ số trùng
    115: 151,
    1221: 2112,
    12443322: 13222344,
    230241: 230412,
    534976: 536479,
}
for n, expected in edge.items():
    got = Solution().nextGreaterElement(n)
    assert got == expected, (n, got, expected)
print("edge OK:", len(edge), "ca")

edge OK: 18 ca


In [22]:
# 3) Đối chiếu: vét cạn (Cách 1) với Cách 2, 3, 4 trên MỌI n từ 1 đến 20000
for n in range(1, 20001):
    a = next_greater_scan(n)
    assert a == next_greater_perms(n) == next_greater_count(n) == next_greater_perm(n) == Solution().nextGreaterElement(n), n
print("1..20000: cả 5 cách giống nhau")

# 4) Ngẫu nhiên có seed, đúng ràng buộc 1 <= n <= 2**31 - 1
def distinct_perms(counter, length):
    # sinh mọi hoán vị KHÁC NHAU của đa tập chữ số (oracle độc lập, không dùng itertools)
    if length == 0:
        yield ""
        return
    for ch in sorted(counter):
        if counter[ch]:
            counter[ch] -= 1
            for rest in distinct_perms(counter, length - 1):
                yield ch + rest
            counter[ch] += 1

def oracle(n):
    s = str(n)
    best = -1
    for t in distinct_perms(Counter(s), len(s)):
        m = int(t)
        if n < m <= INT_MAX and (best == -1 or m < best):
            best = m
    return best

rng = random.Random(556)
cnt_checked = 0
for _ in range(1500):
    d = rng.randint(1, 10)
    if d == 10:
        n = rng.randint(10 ** 9, INT_MAX)
        # lấy ít loại chữ số (<= 3 loại) để oracle vẫn nhanh
        digs = rng.sample("0123456789", rng.randint(1, 3))
        s = "".join(rng.choice(digs) for _ in range(10))
        if s[0] == "0" or int(s) > INT_MAX:
            continue
        n = int(s)
    else:
        n = rng.randint(10 ** (d - 1), 10 ** d - 1)
        if d >= 9:
            digs = rng.sample("0123456789", rng.randint(1, 4))
            s = "".join(rng.choice(digs) for _ in range(d))
            if s[0] == "0":
                continue
            n = int(s)
    exp = oracle(n)
    assert Solution().nextGreaterElement(n) == exp == next_greater_count(n), n
    cnt_checked += 1
print("ngẫu nhiên:", cnt_checked, "số, 3 cách khớp oracle độc lập")

1..20000: cả 5 cách giống nhau


ngẫu nhiên: 1348 số, 3 cách khớp oracle độc lập


In [23]:
# 5) Kết quả luôn là hoán vị của n, lớn hơn n, và không có số hợp lệ nào nằm giữa (kiểm với n nhỏ)
rng = random.Random(7)
for _ in range(2000):
    n = rng.randint(1, 99999)
    r = Solution().nextGreaterElement(n)
    if r == -1:
        assert all(m <= n or sorted(str(m)) != sorted(str(n)) for m in range(n + 1, 10 ** len(str(n))))
    else:
        assert r > n and sorted(str(r)) == sorted(str(n))
        assert all(sorted(str(m)) != sorted(str(n)) for m in range(n + 1, r))
print("tính chất: hoán vị, lớn hơn, nhỏ nhất, và -1 khi không còn số nào")

tính chất: hoán vị, lớn hơn, nhỏ nhất, và -1 khi không còn số nào


## 13. Các lỗi người mới hay gặp

### Lỗi 1: quên kiểm tra tràn 32-bit

Python không tràn số nên `9199999999` là số hoàn toàn hợp lệ trong Python, nhưng đề **yêu cầu** `-1` khi vượt `2**31 - 1`.

In [24]:
def wrong_no_overflow_check(n):
    cs = list(str(n))
    i = len(cs) - 2
    while i >= 0 and cs[i] >= cs[i + 1]:
        i -= 1
    if i < 0:
        return -1
    j = len(cs) - 1
    while cs[j] <= cs[i]:
        j -= 1
    cs[i], cs[j] = cs[j], cs[i]
    cs[i + 1:] = cs[i + 1:][::-1]
    return int("".join(cs))                 # SAI: thiếu kiểm tra tràn

print("sai :", wrong_no_overflow_check(1999999999), "| đúng:", Solution().nextGreaterElement(1999999999))

sai : 9199999999 | đúng: -1


**Vì sao sai:** thiếu dòng `return ans if ans <= 2**31 - 1 else -1`. Hãy luôn đọc kỹ ghi chú trong đề (Note) trước khi code. **Code đúng** chính là Cách 4 phía trên.

### Lỗi 2: dùng `>` thay vì `>=` khi tìm pivot

Nếu viết `cs[i] > cs[i + 1]` thì hai chữ số **bằng nhau** bị coi như "đã tăng được", pivot sẽ dừng nhầm ở chỗ có cặp chữ số trùng.

In [25]:
def wrong_strict(n):
    cs = list(str(n))
    i = len(cs) - 2
    while i >= 0 and cs[i] > cs[i + 1]:     # SAI: phải là >=
        i -= 1
    if i < 0:
        return -1
    j = len(cs) - 1
    while j > i and cs[j] <= cs[i]:
        j -= 1
    cs[i], cs[j] = cs[j], cs[i]
    cs[i + 1:] = cs[i + 1:][::-1]
    ans = int("".join(cs))
    return ans if ans <= INT_MAX else -1

bad = [n for n in range(1, 2000) if wrong_strict(n) != next_greater_perm(n)]
print("số n < 2000 mà code sai trả khác:", len(bad), "ví dụ:", bad[:6])
for n in bad[:3]:
    print(f"  n={n}: sai trả {wrong_strict(n)}, đúng trả {next_greater_perm(n)}")

số n < 2000 mà code sai trả khác: 289 ví dụ: [11, 22, 33, 44, 55, 66]
  n=11: sai trả 11, đúng trả -1
  n=22: sai trả 22, đúng trả -1
  n=33: sai trả 33, đúng trả -1


**Vì sao sai:** với `n = 11`, `cs[0] > cs[1]` là `False` nên pivot dừng ở `0` dù `1` không nhỏ hơn `1`; việc tìm `j` rồi không có chữ số lớn hơn, kết quả trả lại chính `11` (không lớn hơn `n`). **Code đúng:** dùng `>=` để bỏ qua chữ số bằng nhau (Cách 4).

### Lỗi 3: chọn sai chữ số để hoán đổi (lấy chữ số lớn hơn *đầu tiên từ trái*)

Hậu tố không tăng nên chữ số lớn hơn `cs[i]` **đầu tiên từ trái** là chữ số **lớn nhất** trong các chữ số lớn hơn `cs[i]` — không phải nhỏ nhất. Ta cần quét từ phải.

In [26]:
def wrong_leftmost_j(n):
    cs = list(str(n))
    i = len(cs) - 2
    while i >= 0 and cs[i] >= cs[i + 1]:
        i -= 1
    if i < 0:
        return -1
    # SAI: lấy chữ số lớn hơn đầu tiên từ trái (chữ số lớn nhất)
    j = next(k for k in range(i + 1, len(cs)) if cs[k] > cs[i])
    cs[i], cs[j] = cs[j], cs[i]
    cs[i + 1:] = cs[i + 1:][::-1]
    ans = int("".join(cs))
    return ans if ans <= INT_MAX else -1

bad = [n for n in range(1, 20000) if wrong_leftmost_j(n) != next_greater_perm(n)]
print("số sai:", len(bad), "ví dụ:", bad[:5])
distinct = [n for n in bad if len(set(str(n))) == len(str(n))]
for n in [bad[0], distinct[0]]:
    print(f"n={n}: sai -> {wrong_leftmost_j(n)}, đúng -> {next_greater_perm(n)}")

số sai: 5703 ví dụ: [122, 132, 133, 142, 143]
n=122: sai -> 221, đúng -> 212
n=132: sai -> 321, đúng -> 213


**Vì sao sai:** với `n = 132`, pivot là `1`, hậu tố `3 2`; chữ số lớn hơn `1` đầu tiên từ trái là `3` (lớn nhất), cho `321`, trong khi đáp án đúng cần chữ số `2` (nhỏ nhất lớn hơn `1`), cho `213`. Với chữ số trùng (ví dụ `122`) việc chọn `j` không phải phải nhất còn làm hậu tố sau hoán đổi không còn "không tăng", nên bước đảo không cho kết quả nhỏ nhất. **Code đúng:** `j` quét từ **phải** với `while cs[j] <= cs[i]: j -= 1`.

### Lỗi 4: quên đảo hậu tố

Sau hoán đổi, hậu tố vẫn **không tăng** (lớn nhất). Nếu không đảo thành tăng dần thì số tạo ra không phải nhỏ nhất.

In [27]:
def wrong_no_reverse(n):
    cs = list(str(n))
    i = len(cs) - 2
    while i >= 0 and cs[i] >= cs[i + 1]:
        i -= 1
    if i < 0:
        return -1
    j = len(cs) - 1
    while cs[j] <= cs[i]:
        j -= 1
    cs[i], cs[j] = cs[j], cs[i]
    # SAI: thiếu cs[i + 1:] = cs[i + 1:][::-1]
    ans = int("".join(cs))
    return ans if ans <= INT_MAX else -1

for n in [12443322, 158476531, 230241]:
    print(f"n={n}: sai {wrong_no_reverse(n)} | đúng {next_greater_perm(n)}")

n=12443322: sai 13443222 | đúng 13222344
n=158476531: sai 158576431 | đúng 158513467
n=230241: sai 230421 | đúng 230412


**Vì sao sai:** số `13443222` (sai) lớn hơn `13222344` (đúng) — ta trả một số hợp lệ nhưng **không nhỏ nhất**. **Code đúng:** thêm dòng đảo hậu tố.

### Lỗi 5: quên trường hợp `-1` (không có pivot)

Nếu bỏ `if i < 0: return -1` thì `cs[i]` với `i = -1` sẽ lấy phần tử **cuối** của danh sách (chỉ số âm trong Python!) và code chạy tiếp ra một kết quả vô nghĩa thay vì báo lỗi rõ ràng.

In [28]:
def wrong_no_minus1(n):
    cs = list(str(n))
    i = len(cs) - 2
    while i >= 0 and cs[i] >= cs[i + 1]:
        i -= 1
    # SAI: không kiểm tra i < 0
    try:
        j = len(cs) - 1
        while cs[j] <= cs[i]:
            j -= 1
        cs[i], cs[j] = cs[j], cs[i]
        cs[i + 1:] = cs[i + 1:][::-1]
        return int("".join(cs))
    except Exception as e:
        return "lỗi: " + repr(e)

print(wrong_no_minus1(4321), "| đúng:", Solution().nextGreaterElement(4321))
print(wrong_no_minus1(21), "| đúng:", Solution().nextGreaterElement(21))

2134 | đúng: -1
21 | đúng: -1


**Vì sao sai:** với `i = -1`, `cs[i]` là chữ số cuối; kết quả hoàn toàn sai (ví dụ `4321` bị trả thành `2134`, trong khi đáp án đúng là `-1`) nhưng không có exception nào cảnh báo. **Code đúng:** luôn kiểm tra `if i < 0: return -1` ngay sau vòng quét pivot.

## 14. Mẹo & nhận dạng pattern

- Thấy "số nhỏ nhất lớn hơn `n` **dùng cùng các chữ số**" hoặc "hoán vị kế tiếp theo thứ tự từ điển" → nghĩ ngay tới **next permutation** (LeetCode 31).
- Mẹo nhớ thứ tự thao tác: **tìm cái tụt (pivot) từ phải → tìm cái "vừa đủ lớn" từ phải → đổi chỗ → đảo đuôi**.
- Luôn dùng `>=` / `<=` ở các vòng quét để chịu được chữ số trùng.
- Số chữ số nhỏ (`<= 10`) nên chuyển sang chuỗi/list là chuyện bình thường; không cần xử lý bằng `% 10`, `// 10`.
- Chú ý đề có ràng buộc "vừa 32-bit" → với Python phải **tự kiểm tra** ở bước cuối.
- Muốn đáp án **trước đó** (số lớn nhất nhỏ hơn `n`) thì đối xứng: tìm vị trí `cs[i] > cs[i+1]` ... xem mục 15.
- Dùng `itertools.permutations` + `set` chỉ nên là **oracle** để kiểm thử với `d <= 8`.

## 15. Biến thể

### 15.1. Hoán vị kế tiếp trên danh sách (LeetCode 31 — Next Permutation)

Cùng thuật toán nhưng làm **tại chỗ** trên mảng số nguyên (không có giới hạn 32-bit; khi không có hoán vị kế tiếp thì đưa về dạng tăng dần).

In [29]:
def next_permutation_inplace(a):
    i = len(a) - 2
    while i >= 0 and a[i] >= a[i + 1]:
        i -= 1
    if i >= 0:
        j = len(a) - 1
        while a[j] <= a[i]:
            j -= 1
        a[i], a[j] = a[j], a[i]
    a[i + 1:] = a[i + 1:][::-1]
    return a

print(next_permutation_inplace([1, 2, 3]))     # [1, 3, 2]
print(next_permutation_inplace([3, 2, 1]))     # [1, 2, 3] (quay vòng)
print(next_permutation_inplace([1, 1, 5]))     # [1, 5, 1]

[1, 3, 2]
[1, 2, 3]
[1, 5, 1]


### 15.2. Số lớn nhất **nhỏ hơn** `n` dùng cùng các chữ số (hoán vị trước đó)

Đối xứng: tìm pivot phải nhất có `cs[i] > cs[i + 1]` (hậu tố không giảm = nhỏ nhất của nó), hoán đổi với chữ số **lớn nhất nhỏ hơn** `cs[i]` ở phải nhất, rồi **sắp hậu tố giảm dần**. Cần thêm điều kiện không bắt đầu bằng `0` (nếu muốn).

In [30]:
def prev_smaller(n):
    cs = list(str(n))
    i = len(cs) - 2
    while i >= 0 and cs[i] <= cs[i + 1]:
        i -= 1
    if i < 0:
        return -1
    j = len(cs) - 1
    while cs[j] >= cs[i]:
        j -= 1
    cs[i], cs[j] = cs[j], cs[i]
    cs[i + 1:] = cs[i + 1:][::-1]
    if cs[0] == "0":
        return -1
    return int("".join(cs))

for n in [21, 12, 1243, 230241, 4321, 1000]:
    exp = max([m for m in all_perms_sorted(n) if m < n and len(str(m)) == len(str(n))], default=-1)
    print(n, "->", prev_smaller(n), "| vét cạn:", exp)
    assert prev_smaller(n) == exp

21 -> 12 | vét cạn: 12
12 -> -1 | vét cạn: -1
1243 -> 1234 | vét cạn: 1234
230241 -> 230214 | vét cạn: 230214
4321 -> 4312 | vét cạn: 4312
1000 -> -1 | vét cạn: -1


### 15.3. Hoán vị thứ `k` tiếp theo

Gọi next permutation `k` lần. Với `k` nhỏ cách này đủ; với `k` lớn cần dùng hệ số giai thừa (factorial number system) — bài LeetCode 60 *Permutation Sequence* đi theo hướng đó.

### 15.4. Một số bài liên quan

- LeetCode 31 Next Permutation (mảng, tại chỗ).
- LeetCode 496/503 Next Greater Element I / II: *cùng tên nhưng khác thuật toán* (ngăn xếp đơn điệu).
- LeetCode 1053 Previous Permutation With One Swap.
- LeetCode 60 Permutation Sequence.

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1 Thử `n+1, n+2, ...` | thử từng số, so tập chữ số | `O(10^d · d log d)` | `O(d)` | chỉ để hiểu đề, `d <= 5` |
| 2 Sinh hoán vị | liệt kê mọi hoán vị rồi lọc | `O(d! · d)` | `O(d! · d)` | làm oracle kiểm thử, `d <= 8` |
| 3 Đếm chữ số tham lam | mảng đếm hậu tố, chọn `e` nhỏ nhất `> d` | `O(d)` | `O(d)` | khi muốn tránh nghĩ về tính đơn điệu của hậu tố |
| 4 Next permutation | pivot, hoán đổi, đảo hậu tố | `O(d)` | `O(d)` | **cách nộp**, ngắn gọn, mở rộng được cho mảng bất kỳ |

## 17. Bài tập vận dụng

Làm lần lượt từ dễ đến khó. Mỗi bài có một ô để bạn tự làm; đáp án ở mục 19, gợi ý ở mục 18.

### Bài 1 (đoán output)

Hàm `mystery(n)` dưới đây trả về gì với từng giá trị? Hãy tự nghĩ rồi điền vào `ex1_answer` (danh sách 4 số), **không** chạy hàm trước khi đoán.

In [31]:
def mystery(n):
    cs = list(str(n))
    i = len(cs) - 2
    while i >= 0 and cs[i] >= cs[i + 1]:
        i -= 1
    return i

# Đoán mystery(4321), mystery(1234), mystery(1243), mystery(115) rồi điền (ví dụ [0, 0, 0, 0])
ex1_answer = None

### Bài 2 (tính tay)

Tính tay (không chạy code) đáp án của `nextGreaterElement` với: `n = 534976`, `n = 1432`, `n = 2199`, `n = 9876543`. Điền vào `ex2_answer` theo thứ tự (dùng `-1` nếu không có).

In [32]:
ex2_answer = None    # ví dụ [0, 0, 0, 0]

### Bài 3 (điền TODO)

Hoàn thành hàm `ex3_next_greater`: các chỗ `TODO` là hai điều kiện của vòng `while`. Cell này **không gọi** hàm; hãy tự điền rồi chạy kiểm tra ở mục 19 (hoặc tự thử).

In [33]:
def ex3_next_greater(n):
    cs = list(str(n))
    i = len(cs) - 2
    # TODO 1: điều kiện lùi pivot
    while i >= 0 and False:
        i -= 1
    if i < 0:
        return -1
    j = len(cs) - 1
    # TODO 2: điều kiện lùi j
    while False:
        j -= 1
    cs[i], cs[j] = cs[j], cs[i]
    cs[i + 1:] = cs[i + 1:][::-1]
    ans = int("".join(cs))
    return ans if ans <= INT_MAX else -1

### Bài 4 (tự viết lại)

Tự viết **không nhìn** hàm `ex4_rewrite(n)` giải bài này theo cách bạn thích (Cách 3 hoặc Cách 4). Để trống `pass` nếu chưa làm.

In [34]:
def ex4_rewrite(n):
    pass

### Bài 5 (biến thể / mini challenge)

Viết `ex5_kth_next(n, k)`: lấy hoán vị kế tiếp của `n` **liên tiếp `k` lần** (số chữ số giữ nguyên, bỏ qua giới hạn 32-bit); nếu giữa chừng hết hoán vị thì trả `-1`. Ví dụ `ex5_kth_next(123, 3) = 231`.

In [35]:
def ex5_kth_next(n, k):
    pass

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: hàm trả về chỉ số `i` cuối vòng lặp, không phải một số.
- Mức 2: `i` là vị trí phải nhất có `cs[i] < cs[i+1]`, hoặc `-1` nếu không có.
- Mức 3: `4321` giảm dần nên `-1`; `1234`: `3 < 4` ở `i = 2`; `1243`: `4 > 3` lùi, `2 < 4` nên `i = 1`; `115`: `1 < 5` ở `i = 1`.

**Bài 2**
- Mức 1: tìm pivot từ phải, rồi chữ số nhỏ nhất lớn hơn nó trong hậu tố.
- Mức 2: sau hoán đổi, sắp hậu tố tăng dần.
- Mức 3: `534976`: pivot `4` (4 < 9) → hoán đổi với `6` → `536974` → đảo hậu tố `974` thành `479` → `536479`. Với `1432`: pivot `1`; hoán đổi với `2` → `2431` → hậu tố `431` thành `134` → `2134`.

**Bài 3**
- Mức 1: pivot lùi khi chữ số hiện tại **không nhỏ hơn** chữ số bên phải.
- Mức 2: `i` lùi khi `cs[i] >= cs[i + 1]`; `j` lùi khi `cs[j] <= cs[i]`.
- Mức 3: `while i >= 0 and cs[i] >= cs[i + 1]:` và `while cs[j] <= cs[i]:`.

**Bài 4**
- Mức 1: viết ra 4 bước bằng lời trước.
- Mức 2: nhớ `-1` khi không có pivot và kiểm tra tràn `2**31 - 1`.
- Mức 3: đối chiếu với `next_greater_perms` trên `n < 5000`.

**Bài 5**
- Mức 1: lặp `k` lần, mỗi lần áp dụng next permutation.
- Mức 2: dùng lại `next_permutation_inplace` ở mục 15 nhưng phải nhận biết khi hết hoán vị (kết quả quay về tăng dần).
- Mức 3: viết hàm một bước `step(n)` trả `-1` khi không có pivot (không kiểm tra 32-bit), rồi `for _ in range(k)`.

## 19. Đáp án tham khảo

### Đáp án Bài 1

In [36]:
ex1_solution = [-1, 2, 1, 1]
assert [mystery(4321), mystery(1234), mystery(1243), mystery(115)] == ex1_solution
print("Bài 1 OK:", ex1_solution)

Bài 1 OK: [-1, 2, 1, 1]


### Đáp án Bài 2

In [37]:
ex2_solution = [536479, 2134, 2919, -1]
for n, e in zip([534976, 1432, 2199, 9876543], ex2_solution):
    assert Solution().nextGreaterElement(n) == e, (n, e)
print("Bài 2 OK:", ex2_solution)

Bài 2 OK: [536479, 2134, 2919, -1]


Giải thích `2199`: từ phải sang, `9 >= 9` nên lùi; tới `1 < 9` nên pivot là `1` (vị trí 1), hậu tố là `9 9`; chữ số nhỏ nhất lớn hơn `1` là `9` (phải nhất), hoán đổi được `2991`, đảo hậu tố `91` thành `19` được `2919`.

### Đáp án Bài 3

In [38]:
def ex3_solution(n):
    cs = list(str(n))
    i = len(cs) - 2
    while i >= 0 and cs[i] >= cs[i + 1]:
        i -= 1
    if i < 0:
        return -1
    j = len(cs) - 1
    while cs[j] <= cs[i]:
        j -= 1
    cs[i], cs[j] = cs[j], cs[i]
    cs[i + 1:] = cs[i + 1:][::-1]
    ans = int("".join(cs))
    return ans if ans <= INT_MAX else -1

for n in range(1, 3000):
    assert ex3_solution(n) == next_greater_perms(n), n
assert ex3_solution(2147483647) == -1
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [39]:
def ex4_solution(n):
    s = str(n)
    cnt = [0] * 10
    for i in range(len(s) - 1, -1, -1):
        d = int(s[i])
        for e in range(d + 1, 10):
            if cnt[e]:
                cnt[e] -= 1
                cnt[d] += 1
                tail = "".join(str(k) * cnt[k] for k in range(10))
                ans = int(s[:i] + str(e) + tail)
                return ans if ans <= INT_MAX else -1
        cnt[d] += 1
    return -1

rng = random.Random(4)
for n in list(range(1, 3000)) + [rng.randint(1, INT_MAX) for _ in range(500)]:
    assert ex4_solution(n) == Solution().nextGreaterElement(n), n
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [40]:
def ex5_solution(n, k):
    cs = list(str(n))
    for _ in range(k):
        i = len(cs) - 2
        while i >= 0 and cs[i] >= cs[i + 1]:
            i -= 1
        if i < 0:
            return -1
        j = len(cs) - 1
        while cs[j] <= cs[i]:
            j -= 1
        cs[i], cs[j] = cs[j], cs[i]
        cs[i + 1:] = cs[i + 1:][::-1]
    return int("".join(cs))

assert ex5_solution(123, 3) == 231
assert ex5_solution(123, 5) == 321
assert ex5_solution(123, 6) == -1
assert ex5_solution(112, 2) == 211
for n in [1243, 12443, 1122]:
    lst = all_perms_sorted(n)
    p = lst.index(n)
    for k in range(0, len(lst) + 2):
        exp = lst[p + k] if p + k < len(lst) else -1
        assert ex5_solution(n, k) == exp, (n, k)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template

```python
class Solution:
    def nextGreaterElement(self, n: int) -> int:
        cs = list(str(n))
        i = len(cs) - 2
        while i >= 0 and cs[i] >= cs[i + 1]:     # 1) pivot: cái đầu tiên "tụt" từ phải
            i -= 1
        if i < 0:
            return -1                              #    toàn dãy không tăng
        j = len(cs) - 1
        while cs[j] <= cs[i]:                      # 2) chữ số vừa đủ lớn hơn từ phải
            j -= 1
        cs[i], cs[j] = cs[j], cs[i]                # 3) đổi chỗ
        cs[i + 1:] = cs[i + 1:][::-1]              # 4) đảo đuôi thành tăng dần
        ans = int("".join(cs))
        return ans if ans <= 2 ** 31 - 1 else -1   # 5) chặn tràn 32-bit
```

Câu thần chú: **"tìm tụt từ phải, tìm vừa-đủ-lớn từ phải, đổi chỗ, đảo đuôi, nhớ chặn 32-bit."**

### Checklist trước khi submit

- [ ] Vòng pivot dùng `>=` (không phải `>`), vòng `j` dùng `<=`.
- [ ] `if i < 0: return -1` có mặt.
- [ ] `j` quét từ **phải** và dừng ở chữ số **lớn hơn hẳn** `cs[i]`.
- [ ] Đã đảo hậu tố `cs[i + 1:]`.
- [ ] Đã kiểm tra `ans > 2**31 - 1` trả `-1`.
- [ ] Test: một chữ số (`1`), toàn chữ số trùng (`11`), giảm dần (`4321`), có số `0` (`101`, `1000000000`), tràn (`1999999999`, `2147483647`).
- [ ] Skeleton đúng: `class Solution`, method `nextGreaterElement(self, n: int) -> int`.